# 18 | LangChain Middleware：限制Agent别疯跑

Agent 会自己调用模型，也会自己调用工具。问题是：**它可能多想几轮，也可能多查几次。**

客服 Agent 查订单，本来查 1 次就够了，结果来回查了 8 次。用户还没等急，账单先急了。

这篇只讲一件事：用 Middleware 限制 Agent 的调用次数。

本篇用到两个官方 Middleware：

- `ModelCallLimitMiddleware`：限制模型调用次数
- `ToolCallLimitMiddleware`：限制工具调用次数

## 一、先准备一个客服 Agent

场景很简单：用户问订单退款，Agent 需要查订单状态，再查退款规则。

这两个工具在真实项目里通常对应订单系统和规则系统。这里用本地数据模拟，重点看 Middleware 怎么限制调用次数。

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool

load_dotenv(dotenv_path=".env")

# 模型需要支持 tool calling，否则 Agent 无法稳定调用工具。
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
)

ORDERS = {
    "O-1001": {"item": "蓝牙耳机", "status": "未发货"},
    "O-1002": {"item": "机械键盘", "status": "已发货"},
}

POLICIES = {
    "未发货": "可以直接退款，1 到 3 个工作日原路退回。",
    "已发货": "需要先退货，验收通过后 3 到 7 个工作日原路退回。",
}


@tool
def get_order_status(order_id: str) -> dict:
    """根据订单号查询订单状态。"""
    # 外部查询类工具最需要控制次数，因为它背后可能是数据库或第三方 API。
    return {"order_id": order_id, **ORDERS.get(order_id, {})}


@tool
def get_refund_policy(order_status: str) -> str:
    """根据订单状态查询退款规则。"""
    return POLICIES.get(order_status, "没有匹配规则，需要转人工。")


tools = [get_order_status, get_refund_policy]


In [ ]:
from langchain.agents import create_agent

system_prompt = (
    "你是一个电商售后 Agent。"
    "回答退款问题前，必须先查询订单状态，再查询退款规则。"
    "回答要简洁说明是否能退、依据和到账时间。"
)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt,
)


没有限制时，Agent 会自己决定调用多少次模型、多少次工具。多数时候没问题，但一旦它开始反复确认、反复查询，就会浪费时间和成本。

所以我们给它加一道边界：可以查，但别跑太远。

## 二、限制模型调用次数

普通退款查询通常不会反复调用模型，演示效果不明显。

我们换一个更容易“跑起来”的场景：用户让 Agent **持续查询退款进度，直到退款完成**。

如果工具一直返回“处理中”，Agent 就可能进入循环：

```text
模型判断：继续查
工具返回：处理中
模型判断：再查一次
工具返回：处理中
...
```

查物流、查审批、查任务状态、查退款进度，都会遇到这种情况。Agent 很努力，但账单也很努力。

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware

# 模拟退款进度接口：前几次都返回“处理中”。
# 真实项目里，这通常是订单系统、支付系统或工单系统的查询接口。
refund_progress_checks = {"O-1001": 0}


@tool
def check_refund_progress(order_id: str) -> str:
    """查询退款进度，如果还没完成，可以稍后再次查询。"""
    refund_progress_checks[order_id] = refund_progress_checks.get(order_id, 0) + 1
    count = refund_progress_checks[order_id]

    if count < 5:
        return f"第 {count} 次查询：退款仍在处理中。"
    return f"第 {count} 次查询：退款已完成。"


loop_tools = [check_refund_progress]

progress_agent = create_agent(
    model=model,
    tools=loop_tools,
    middleware=[
        # 单次 invoke 最多允许 3 次模型调用。
        # 即使工具一直返回“处理中”，Agent 也不能无限判断下去。
        ModelCallLimitMiddleware(run_limit=3, exit_behavior="end"),
    ],
    system_prompt=(
        "你是一个退款进度查询 Agent。"
        "如果退款还在处理中，就继续调用 check_refund_progress 查询。"
        "如果达到系统限制，就基于已有结果简洁说明当前状态。"
    ),
)


In [ ]:
response = progress_agent.invoke({
    "messages": [
        {"role": "user", "content": "帮我一直查订单 O-1001 的退款进度，直到退款完成。"}
    ]
})

print(response["messages"][-1].content)


如果不加 `ModelCallLimitMiddleware`，Agent 可能一直查到第 5 次，直到工具返回“退款已完成”。

但我们设置了：

```python
ModelCallLimitMiddleware(run_limit=3, exit_behavior="end")
```

意思是：**单次请求最多调用 3 次模型。**

所以当 Agent 还想继续判断下一步时，会被 Middleware 拦住。

运行后你会看到类似结果：

```text
Model call limits exceeded: run limit (3/3)
```

这不是代码错了，而是限制生效了。

这里故意把 `run_limit` 设得比较小，是为了让效果更明显。真实业务里要根据任务复杂度调整，别小到 Agent 还没开始干活就被叫停。

## 三、限制工具调用次数

上一节限制的是“模型最多判断几轮”。

但真正打到业务系统上的，是 `check_refund_progress` 这个工具。它背后可能连着支付系统、订单系统或第三方接口，所以工具调用次数也要管。

这里限制退款进度查询工具：单次请求最多查 2 次。

In [ ]:
from langchain.agents.middleware import ToolCallLimitMiddleware

# 重置查询次数，方便观察工具调用限制的效果。
refund_progress_checks = {"O-1001": 0}

tool_limited_progress_agent = create_agent(
    model=model,
    tools=loop_tools,
    middleware=[
        # 只限制 check_refund_progress：单次 invoke 最多调用 2 次。
        # 即使 Agent 还想继续查，也不能继续打这个工具。
        ToolCallLimitMiddleware(
            tool_name="check_refund_progress",
            run_limit=2,
            exit_behavior="continue",
        ),
    ],
    system_prompt=(
        "你是一个退款进度查询 Agent。"
        "如果退款还在处理中，就继续调用 check_refund_progress 查询。"
        "如果工具调用被限制，就基于已有查询结果说明当前状态。"
    ),
)


In [ ]:
response = tool_limited_progress_agent.invoke({
    "messages": [
        {"role": "user", "content": "帮我一直查订单 O-1001 的退款进度，直到退款完成。"}
    ]
})

print(response["messages"][-1].content)


添加限制后，Agent 最多只能查两次退款进度。即使它还想继续查，Middleware 也会拦住第 3 次工具调用。

输出大概会变成：

```text
已查询两次，退款仍在处理中。
由于工具调用次数已达到限制，建议稍后再查或转人工确认。
```

如果不加限制，Agent 可能会一直查到第 5 次，直到返回“退款已完成”。

这就是工具调用限制的意义：不是不让 Agent 干活，而是防止它反复刷同一个外部接口。

这两类限制可以这样理解：

```text
ModelCallLimitMiddleware：限制 Agent 想几轮
ToolCallLimitMiddleware：限制 Agent 动几次手
```

客服场景里，查订单、查退款、查物流这些工具，本质上都在消耗业务系统资源。让 Agent 查一次两次可以，来回刷接口就不太礼貌了。

## 四、run_limit 和 thread_limit 的区别

这两个参数很容易混：

| 参数 | 控制范围 | 适合场景 |
|---|---|---|
| `run_limit` | 单次 `invoke()` | 限制一次用户请求内最多调用几次 |
| `thread_limit` | 同一个会话线程 | 限制整个对话累计最多调用几次 |

举个客服场景：

- 用户这一句话里，最多查订单 1 次：用 `run_limit`
- 同一个用户一整轮对话里，最多查订单 5 次：用 `thread_limit`

`thread_limit` 需要保存会话状态，所以通常要配合 `checkpointer`。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

thread_limited_agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=checkpointer,
    middleware=[
        # 同一个 thread_id 下，累计最多调用 6 次模型；单次请求最多调用 3 次模型。
        ModelCallLimitMiddleware(thread_limit=6, run_limit=3, exit_behavior="end"),

        # 同一个 thread_id 下，订单查询累计最多 3 次。
        ToolCallLimitMiddleware(tool_name="get_order_status", thread_limit=3, run_limit=1),
    ],
    system_prompt=system_prompt,
)

config = {"configurable": {"thread_id": "customer-support-001"}}


In [ ]:
# 第一次请求和第二次请求使用同一个 thread_id。
# thread_limit 会跨多次 invoke 累计计算。
response = thread_limited_agent.invoke(
    {"messages": [{"role": "user", "content": "订单 O-1001 可以退款吗？"}]},
    config=config,
)

print(response["messages"][-1].content)


如果只是本地跑一次 demo，`run_limit` 更直观。

如果是连续对话、客服会话、工单处理这类真实业务，`thread_limit` 更重要。因为用户可能分几次问同一个问题，Agent 也可能分几次重复查同一个接口。

说得直白一点：单次重复查叫浪费，连续多轮重复查就有点像小型压测了。

## 五、超限后：结束还是报错

达到限制以后，Middleware 需要决定怎么处理。

`ModelCallLimitMiddleware` 支持：

- `exit_behavior="end"`：尽量正常结束
- `exit_behavior="error"`：直接抛异常

`ToolCallLimitMiddleware` 支持：

- `exit_behavior="continue"`：默认，把超限信息交给模型，让模型继续处理
- `exit_behavior="error"`：直接抛异常
- `exit_behavior="end"`：适合限制单个工具时直接结束

怎么选？

- 面向用户的客服回答，通常用 `end` 或 `continue`，体验更平滑。
- 内部任务、测试、审计场景，可以用 `error`，问题暴露得更直接。
- 高风险工具，比如退款、删除、发送邮件，宁愿 `error`，也别悄悄继续。

In [ ]:
strict_agent = create_agent(
    model=model,
    tools=tools,
    middleware=[
        # 达到模型调用上限后直接抛异常，适合测试或高风险流程。
        ModelCallLimitMiddleware(run_limit=1, exit_behavior="error"),
    ],
    system_prompt=system_prompt,
)

try:
    strict_agent.invoke({
        "messages": [
            {"role": "user", "content": "订单 O-1001 可以退款吗？请查清楚再回答。"}
        ]
    })
except Exception as e:
    print(type(e).__name__)
    print(e)


如果只记一句话：**`run_limit` 管这一次，`thread_limit` 管这一整段会话；`end` 适合温和收尾，`error` 适合硬性拦截。**

下一篇继续看失败兜底：模型或工具真的失败时，怎么自动重试、怎么切备用模型。

参考：

- LangChain Prebuilt Middleware：https://docs.langchain.com/oss/python/langchain/middleware/built-in
- LangChain Middleware Overview：https://docs.langchain.com/oss/python/langchain/middleware/overview